# 🚀 Kashmiri AI Backend Server: Whisper Speech + NLLB Translation (Google Colab T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sameerahmad86232/cipher/blob/main/colab-kashmiri-ai-server.ipynb)

This notebook runs the complete **Kashmiri AI Inference Server** on Google Colab's **Free NVIDIA Tesla T4 GPU**:
- **Speech-to-Text**: `Sameer0313/kashmiri-whisper-lora` (Whisper Large-V3-Turbo, CER: 22.42%)
- **Machine Translation**: `Sameer0313/kashmiri-nllb-200-master-lora` (NLLB-200 320k Pairs, Loss: 2.02)
- **Public API**: Launches a free, secure `gradio.live` public URL that connects directly to your web app (`sameerahmad86232/cipher`)!

**Hardware Recommendation:** Runtime -> Change runtime type -> select **T4 GPU**.

In [ ]:
# Step 1: Install Dependencies
!pip install -q transformers peft soundfile librosa gradio accelerate

In [ ]:
# Step 2: Load Kashmiri AI Models on GPU
import torch
import librosa
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[+] Computation Device: {DEVICE}")

# 1. Load Kashmiri Whisper Large-V3-Turbo
WHISPER_BASE = "openai/whisper-large-v3-turbo"
WHISPER_ADAPTER = "Sameer0313/kashmiri-whisper-lora"
print("[*] Loading Whisper Speech Processor & LoRA Adapter...")
whisper_processor = WhisperProcessor.from_pretrained(WHISPER_BASE, language="urdu", task="transcribe")
whisper_base = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_BASE,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto"
)
whisper_model = PeftModel.from_pretrained(whisper_base, WHISPER_ADAPTER)
whisper_model.eval()
print("[+] Whisper Speech-to-Text Model is READY!")

# 2. Load Master Kashmiri NLLB-200 Translation
NLLB_BASE = "facebook/nllb-200-distilled-600M"
NLLB_ADAPTER = "Sameer0313/kashmiri-nllb-200-master-lora"
print("[*] Loading Master NLLB Translation Model & LoRA Adapter...")
nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_BASE)
nllb_base = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_BASE,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto"
)
nllb_model = PeftModel.from_pretrained(nllb_base, NLLB_ADAPTER)
nllb_model.eval()
print("[+] Master NLLB Translation Model is READY!")

In [ ]:
# Step 3: Define Inference Endpoints (Matching Web App API Schema)

def transcribe_kashmiri(audio_path):
    if not audio_path:
        return ""
    speech, sr = librosa.load(audio_path, sr=16000)
    inputs = whisper_processor(speech, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to(whisper_model.device, dtype=whisper_model.dtype)
    with torch.no_grad():
        predicted_ids = whisper_model.generate(
            inputs,
            num_beams=2,
            max_new_tokens=225
        )
    text = whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return text.strip()

def translate(text, direction="Kashmiri → English"):
    if not text or not text.strip():
        return "", ""
    src = "kas_Arab" if "Kashmiri" in direction else "eng_Latn"
    tgt = "eng_Latn" if "Kashmiri" in direction else "kas_Arab"
    nllb_tokenizer.src_lang = src
    inputs = nllb_tokenizer(text, return_tensors="pt").to(nllb_model.device)
    with torch.no_grad():
        generated_tokens = nllb_model.generate(
            **inputs,
            forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids(tgt),
            max_new_tokens=128,
            num_beams=4
        )
    result = nllb_tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    return result.strip(), text.strip()

print("[+] Inference functions verified!")

In [ ]:
# Step 4: Launch Public Server (With Gradio Live Tunnel)
import gradio as gr

with gr.Blocks(title="Koshur Lughat AI Server") as demo:
    gr.Markdown("# 🌟 Koshur Lughat Kashmiri AI Backend Server")
    gr.Markdown("Powered by fine-tuned **Whisper Large-V3-Turbo** & **Master NLLB-200** on Google Colab GPU.")
    
    with gr.Tab("Speech-to-Text"):
        audio_in = gr.Audio(type="filepath", label="Kashmiri Audio Input (Mic or File)")
        text_out = gr.Textbox(label="Perso-Arabic Kashmiri Output")
        btn_transcribe = gr.Button("Transcribe Kashmiri Speech")
        btn_transcribe.click(
            transcribe_kashmiri,
            inputs=audio_in,
            outputs=text_out,
            api_name="transcribe_kashmiri"
        )
        
    with gr.Tab("Neural Translation"):
        text_in = gr.Textbox(label="Text to Translate")
        dir_in = gr.Dropdown(
            choices=["Kashmiri → English", "English → Kashmiri"],
            value="Kashmiri → English",
            label="Translation Direction"
        )
        trans_out = gr.Textbox(label="Translated Result")
        norm_out = gr.Textbox(label="Normalized Text")
        btn_translate = gr.Button("Translate")
        btn_translate.click(
            translate,
            inputs=[text_in, dir_in],
            outputs=[trans_out, norm_out],
            api_name="translate"
        )

print("=" * 70)
print("Launching Public Gradio URL... Copy the https://xxxx.gradio.live link below")
print("and paste it into the Speech Lab endpoint settings in your React app!")
print("=" * 70)

demo.queue().launch(share=True, debug=True)